# Mobile Suit AMBAC Wireframe Demo

This notebook demonstrates:
- Loading a minimal wireframe mobile suit model in MuJoCo
- Parameterizing thruster positions
- Running a simulation with thruster control
- Rendering frames and creating a GIF

## Important: Run cells in order!
1. **Run cell 1 first** - Installs OSMesa for headless rendering
2. If you get OpenGL errors, **Restart runtime** and run cell 1 first

In [ ]:
# ============================================
# COLAB SETUP - Run this cell FIRST!
# ============================================
import os
import sys

# Must set BEFORE any mujoco import
if 'mujoco' in sys.modules:
    print("ERROR: mujoco already imported!")
    print("Go to Runtime > Restart runtime, then run this cell first.")
    raise SystemExit("Restart required")

# Try EGL first (GPU), fall back to OSMesa (CPU)
# EGL is faster but may not work on all Colab instances
os.environ['MUJOCO_GL'] = 'egl'

# Install dependencies
!pip install -q mujoco dm-control imageio imageio[ffmpeg] matplotlib

# Install rendering libraries
!apt-get update -qq > /dev/null
!apt-get install -qq -y libegl1-mesa-dev libgl1-mesa-dev libgles2-mesa-dev > /dev/null 2>&1
!apt-get install -qq -y libosmesa6-dev > /dev/null 2>&1

# Clone repo
!git clone https://github.com/OsirisRaptor/mobile-suit-sim-mujoco.git 2>/dev/null || echo "Repo exists"
%cd mobile-suit-sim-mujoco

# Test which backend works
print("\nTesting rendering backend...")
try:
    import mujoco
    model = mujoco.MjModel.from_xml_string('<mujoco><worldbody><body><geom size="1"/></body></worldbody></mujoco>')
    renderer = mujoco.Renderer(model, 64, 64)
    renderer.close()
    print(f"SUCCESS: Using MUJOCO_GL={os.environ.get('MUJOCO_GL')}")
except Exception as e:
    print(f"EGL failed: {e}")
    print("Trying OSMesa...")
    # Restart with OSMesa would be needed - inform user
    print("\n*** If rendering fails, try adding this to the TOP of the next cell: ***")
    print("import os; os.environ['MUJOCO_GL'] = 'osmesa'")

In [ ]:
import os
import numpy as np
import mujoco
from mobile_suit_sim.config import ThrusterConfig
from mobile_suit_sim.model_builder import load_model, apply_thruster_config
from mobile_suit_sim.sim_runner import simulate_and_render
from IPython.display import Image, display

# Path to MJCF
mjcf_path = os.path.join("models", "ambac_test.mjcf")

model, data = load_model(mjcf_path)

print("nq (positions):", model.nq)
print("nv (velocities):", model.nv)
print("nbody:", model.nbody)
print("nsite:", model.nsite)

In [ ]:
# Define a single thruster config
thruster_cfgs = [
    ThrusterConfig(
        name="main",          # maps to site "thruster_main"
        body_name="torso",    # not used yet, but good for future checks
        pos=[0.0, -0.6, 0.0], # x, y, z in torso frame
        dir_world=[0.0, -1.0, 0.0],
        max_force=2000.0
    )
]

site_ids = apply_thruster_config(model, thruster_cfgs)
site_ids

In [ ]:
def control_fn(step, model, data):
    # We only have one general actuator: thruster_main_act
    # In general actuators, data.ctrl is in the same order as actuators.
    n_steps_burn = 150

    if step < n_steps_burn:
        data.ctrl[:] = 1500.0  # N, < ctrlrange max
    else:
        data.ctrl[:] = 0.0

In [ ]:
# Try rendering, with fallback to physics-only
try:
    output_gif = simulate_and_render(
        model=model,
        data=data,
        n_steps=300,
        control_fn=control_fn,
        width=480,
        height=360,
        camera="fixed",
        output_path="ambac_thruster_demo.gif"
    )
    display(Image(filename=output_gif))
    
except Exception as e:
    print(f"Rendering failed: {e}")
    print("\n--- Running physics-only simulation instead ---\n")
    
    # Reset and run physics only
    import mujoco
    mujoco.mj_resetData(model, data)
    
    positions = []
    for step in range(300):
        control_fn(step, model, data)
        mujoco.mj_step(model, data)
        if step % 10 == 0:
            positions.append(data.qpos[0:3].copy())
    
    # Plot trajectory
    import matplotlib.pyplot as plt
    positions = np.array(positions)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.plot(positions[:, 0], label='X (forward)')
    ax.plot(positions[:, 1], label='Y (lateral)')
    ax.plot(positions[:, 2], label='Z (vertical)')
    ax.set_xlabel('Time step (x10)')
    ax.set_ylabel('Position (m)')
    ax.set_title('Thruster Simulation - Position over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\nFinal position: {positions[-1]}")
    print(f"Total displacement: {np.linalg.norm(positions[-1] - positions[0]):.2f} m")